### Instalación de dependencias (Opcional, ejecutar si falta alguna librería)

In [3]:
# Ejecuta esta celda únicamente si no tienes instaladas las librerías en tu entorno de Jupyter.
# El signo '!' le indica a Jupyter que ejecute el comando directamente en la terminal del sistema.

!pip install pandas numpy scikit-learn

  Using cached tzdata-2026.2-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
   ---------------------------------------- 0.0/9.8 MB ? eta -:--:--
   ------------------------------ --------- 7.3/9.8 MB 42.0 MB/s eta 0:00:01
   ---------------------------------------- 9.8/9.8 MB 25.0 MB/s  0:00:00
   ---------------------------------------- 0.0/12.3 MB ? eta -:--:--
   ------------------------- -------------- 7.9/12.3 MB 38.2 MB/s eta 0:00:01
   ---------------------------------------  12.1/12.3 MB 36.6 MB/s eta 0:00:01
   ---------------------------------------  12.1/12.3 MB 36.6 MB/s eta 0:00:01
   ---------------------------------------- 12.3/12.3 MB 15.7 MB/s  0:00:00
   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   ------------------------------------ --- 7.3/8.0 MB 35.9 MB/s eta 0:00:01
   ---------------------------------------  7.9/8.0 MB 36.3 MB/s eta 0:00:01
   ---------------------------------------- 

### Importación de librerías necesarias

In [4]:
# Importamos las librerías fundamentales para manipulación de datos,
# modelado estadístico y evaluación de métricas de Machine Learning
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

### Carga de datos desde el repositorio de GitHub

In [5]:
# Definimos la URL del archivo raw provista por el repositorio de GitHub
url_dataset = "https://raw.githubusercontent.com/gmmorales/ifts24-data-science-datasets/refs/heads/master/datasets/modelizado-de-sistemas-de-ia/integrador/dataset_clean.csv"

# Cargamos el dataset directamente a un DataFrame de pandas
print("Cargando datos desde GitHub...")
df = pd.read_csv(url_dataset)

# Visualizamos las primeras filas para verificar que la lectura fue correcta
print(f"Dataset cargado con éxito. Dimensiones originales: {df.shape}")
df.head()

Cargando datos desde GitHub...
Dataset cargado con éxito. Dimensiones originales: (152739, 15)


,id-mapa,anio,mes,dia,fecha,franja,tipo,subtipo,uso_arma,uso_moto,barrio,comuna,latitud,longitud,cantidad
0,1114988,2023,NOVIEMBRE,JUEVES,30/11/2023,17,Amenazas,Amenazas,NO,NO,BOCA,4,-34.625.121,-58.348.646,1
1,1114989,2023,NOVIEMBRE,JUEVES,30/11/2023,17,Amenazas,Amenazas,NO,NO,BOCA,4,-34.625.121,-58.348.646,1
2,1114990,2023,NOVIEMBRE,JUEVES,30/11/2023,17,Amenazas,Amenazas,NO,NO,BOCA,4,-34.625.121,-58.348.646,1
3,1114991,2023,AGOSTO,MARTES,8/8/2023,20,Amenazas,Amenazas,NO,NO,BOCA,4,-34.633.645,-58.353.495,1
4,1114992,2023,JUNIO,LUNES,19/6/2023,16,Amenazas,Amenazas,NO,NO,BOCA,4,-34.634.926,-58.353.819,1


### Preprocesamiento y Agregación de Datos

In [6]:
# El dataset registra incidentes individuales (1 fila = 1 delito).
# Para realizar una regresión que prediga la 'cantidad' (volumen), debemos
# agrupar los datos según nuestras dimensiones de análisis: Comuna y Franja Horaria.

# Nos aseguramos de que las variables clave sean tratadas como tipos numéricos enteros
df['comuna'] = df['comuna'].astype(int)
df['franja'] = pd.to_numeric(df['franja'], errors='coerce')

# Eliminamos posibles filas que hayan quedado con franjas horarias inválidas tras la conversión
df = df.dropna(subset=['franja'])
df['franja'] = df['franja'].astype(int)

# AGREGACIÓN: Agrupamos por comuna y franja y calculamos el volumen total de delitos (target)
# .size() contará la cantidad de filas (delitos) para cada combinación única
df_modelo = df.groupby(['comuna', 'franja']).size().reset_index(name='volumen_delitos')

print("Dataset preparado y consolidado para el modelo de Regresión Lineal:")
df_modelo.head(10)

Dataset preparado y consolidado para el modelo de Regresión Lineal:


,comuna,franja,volumen_delitos
0,1,0,781
1,1,1,470
2,1,2,394
3,1,3,378
4,1,4,431
5,1,5,517
6,1,6,667
7,1,7,723
8,1,8,813
9,1,9,818


### División del dataset (Features vs Target y Train/Test Split)

In [7]:
# Definimos nuestras variables independientes o predictores (X)
# En este escenario base: la dimensión espacial (comuna) y la temporal (franja)
X = df_modelo[['comuna', 'franja']]

# Definimos nuestra variable dependiente continua o el objetivo a predecir (y)
y = df_modelo['volumen_delitos']

# Dividimos el conjunto de datos de manera aleatoria:
# - 80% para el entrenamiento del modelo (Train)
# - 20% para la validación y prueba de métricas (Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Datos de entrenamiento (X_train): {X_train.shape}")
print(f"Datos de evaluación (X_test): {X_test.shape}")

Datos de entrenamiento (X_train): (288, 2)
Datos de evaluación (X_test): (72, 2)


### Entrenamiento del Modelo de Regresión Lineal

In [8]:
# Instanciamos el objeto del modelo de Regresión Lineal de scikit-learn
modelo_lineal = LinearRegression()

# Entrenamos el algoritmo utilizando el set de datos de entrenamiento
# Aquí el modelo calcula matemáticamente la intersección (beta_0) y los coeficientes (betas)
modelo_lineal.fit(X_train, y_train)

print("¡Modelo entrenado exitosamente!")
print(f"Intersección (Beta 0): {modelo_lineal.intercept_:.4f}")
print(f"Coeficientes (Betas para Comuna y Franja): {modelo_lineal.coef_}")

¡Modelo entrenado exitosamente!
Intersección (Beta 0): 372.7843
Coeficientes (Betas para Comuna y Franja): [-12.39001846  13.21597115]


### Predicción y Evaluación de Métricas

In [9]:
# Ejecutamos las predicciones utilizando las características del set de evaluación (X_test)
y_pred = modelo_lineal.predict(X_test)

# Calculamos las métricas estadísticas estándar para evaluar el rendimiento del modelo de regresión
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("==================== EVALUACIÓN DEL MODELO ====================")
print(f"Error Absoluto Medio (MAE): {mae:.2f} delitos")
print(f"Error Cuadrático Medio Mínimo (MSE): {mse:.2f}")
print(f"Raíz del Error Cuadrático Medio (RMSE): {rmse:.2f} delitos")
print(f"Coeficiente de Determinación (R²): {r2:.4f}")
print("===============================================================")

==================== EVALUACIÓN DEL MODELO ====================
Error Absoluto Medio (MAE): 133.05 delitos
Error Cuadrático Medio Mínimo (MSE): 27318.27
Raíz del Error Cuadrático Medio (RMSE): 165.28 delitos
Coeficiente de Determinación (R²): 0.1808
